# 🔤 Tokenizer Lab - Thí Nghiệm Tokenization

## Mục tiêu bài học
- Hiểu cách Large Language Models (LLM) chia nhỏ văn bản thành tokens
- So sánh sự khác biệt giữa tokenization tiếng Việt và tiếng Anh
- Tính toán chi phí sử dụng API dựa trên số lượng tokens
- Tối ưu hóa prompt để tiết kiệm chi phí

## 📚 Phần 1: Giới thiệu về Tokenization

### Tokenization là gì?
Tokenization là quá trình chia nhỏ văn bản thành các đơn vị nhỏ hơn gọi là **tokens**. Đây là bước đầu tiên mà LLM thực hiện khi xử lý văn bản.

### Tại sao cần tokenization?
- LLM không xử lý trực tiếp văn bản, mà xử lý các con số (tokens)
- Mỗi token được chuyển đổi thành một vector số để model có thể hiểu
- Chi phí API được tính dựa trên số lượng tokens (input + output)

### Một số quy tắc cơ bản:
- 1 token ≈ 4 ký tự tiếng Anh
- 1 token ≈ ¾ từ tiếng Anh
- 1 từ tiếng Việt có thể tốn nhiều token hơn tiếng Anh (2-3 tokens)
- Khoảng trắng, dấu câu cũng tốn tokens

## 🛠️ Phần 2: Cài đặt và Chuẩn bị

Chúng ta sẽ sử dụng thư viện `tiktoken` - công cụ tokenization của OpenAI

In [3]:
# Cài đặt thư viện cần thiết
!pip install tiktoken pandas

  Using cached pandas-3.0.0-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.1-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.0-cp311-cp311-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.1-cp311-cp311-win_amd64.whl (12.6 MB)
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Import thư viện
import tiktoken
import pandas as pd
from typing import List

## 🔬 Phần 3: Thử nghiệm Tokenization

### 3.1 Khởi tạo Tokenizer
Chúng ta sẽ sử dụng tokenizer của GPT-4 (encoding: `cl100k_base`)

In [5]:
# Khởi tạo tokenizer cho GPT-4/GPT-3.5-turbo
encoding = tiktoken.get_encoding("cl100k_base")

# Hoặc có thể khởi tạo theo tên model cụ thể
# encoding = tiktoken.encoding_for_model("gpt-4")

print("✅ Tokenizer đã được khởi tạo thành công!")

✅ Tokenizer đã được khởi tạo thành công!


### 3.2 Hàm tiện ích để phân tích tokens

In [6]:
def analyze_text(text: str, encoding) -> dict:
    """
    Phân tích văn bản và trả về thông tin về tokens
    
    Args:
        text: Văn bản cần phân tích
        encoding: Tokenizer encoding
    
    Returns:
        Dictionary chứa thông tin phân tích
    """
    # Tokenize văn bản
    tokens = encoding.encode(text)
    
    # Decode từng token để xem nội dung
    token_strings = [encoding.decode([token]) for token in tokens]
    
    return {
        'text': text,
        'num_tokens': len(tokens),
        'num_characters': len(text),
        'num_words': len(text.split()),
        'tokens': tokens,
        'token_strings': token_strings,
        'chars_per_token': round(len(text) / len(tokens), 2) if len(tokens) > 0 else 0
    }

def display_tokens(text: str, encoding, max_display=20):
    """
    Hiển thị chi tiết tokens của văn bản
    """
    result = analyze_text(text, encoding)
    
    print(f"📝 Văn bản gốc: {result['text'][:100]}..." if len(text) > 100 else f"📝 Văn bản gốc: {result['text']}")
    print(f"\n📊 Thống kê:")
    print(f"   - Số ký tự: {result['num_characters']}")
    print(f"   - Số từ: {result['num_words']}")
    print(f"   - Số tokens: {result['num_tokens']}")
    print(f"   - Trung bình: {result['chars_per_token']} ký tự/token")
    
    print(f"\n🔤 Chi tiết tokens (hiển thị {min(max_display, len(result['token_strings']))} tokens đầu):")
    for i, (token_id, token_str) in enumerate(zip(result['tokens'][:max_display], result['token_strings'][:max_display])):
        # Hiển thị token với ký tự đặc biệt được thay thế để dễ đọc
        display_str = token_str.replace('\n', '\\n').replace('\t', '\\t').replace(' ', '␣')
        print(f"   Token {i+1}: [{token_id:5d}] → '{display_str}'")
    
    if len(result['tokens']) > max_display:
        print(f"   ... và {len(result['tokens']) - max_display} tokens nữa")
    
    return result

### 3.3 Thử nghiệm với tiếng Anh

In [7]:
# Ví dụ tiếng Anh đơn giản
english_text = "Hello, how are you today?"

print("=" * 70)
print("THỬ NGHIỆM TIẾNG ANH")
print("=" * 70)
result_en = display_tokens(english_text, encoding)

THỬ NGHIỆM TIẾNG ANH
📝 Văn bản gốc: Hello, how are you today?

📊 Thống kê:
   - Số ký tự: 25
   - Số từ: 5
   - Số tokens: 7
   - Trung bình: 3.57 ký tự/token

🔤 Chi tiết tokens (hiển thị 7 tokens đầu):
   Token 1: [ 9906] → 'Hello'
   Token 2: [   11] → ','
   Token 3: [ 1268] → '␣how'
   Token 4: [  527] → '␣are'
   Token 5: [  499] → '␣you'
   Token 6: [ 3432] → '␣today'
   Token 7: [   30] → '?'


In [8]:
# Ví dụ tiếng Anh dài hơn
english_paragraph = """
Artificial intelligence is transforming the world. Machine learning models 
can process vast amounts of data and make predictions with remarkable accuracy. 
Large language models like GPT can understand and generate human-like text.
"""

print("=" * 70)
print("THỬ NGHIỆM TIẾNG ANH - ĐOẠN VĂN DÀI")
print("=" * 70)
result_en_long = display_tokens(english_paragraph.strip(), encoding)

THỬ NGHIỆM TIẾNG ANH - ĐOẠN VĂN DÀI
📝 Văn bản gốc: Artificial intelligence is transforming the world. Machine learning models 
can process vast amounts...

📊 Thống kê:
   - Số ký tự: 232
   - Số từ: 32
   - Số tokens: 40
   - Trung bình: 5.8 ký tự/token

🔤 Chi tiết tokens (hiển thị 20 tokens đầu):
   Token 1: [ 9470] → 'Art'
   Token 2: [16895] → 'ificial'
   Token 3: [11478] → '␣intelligence'
   Token 4: [  374] → '␣is'
   Token 5: [46890] → '␣transforming'
   Token 6: [  279] → '␣the'
   Token 7: [ 1917] → '␣world'
   Token 8: [   13] → '.'
   Token 9: [13257] → '␣Machine'
   Token 10: [ 6975] → '␣learning'
   Token 11: [ 4211] → '␣models'
   Token 12: [  720] → '␣\n'
   Token 13: [ 4919] → 'can'
   Token 14: [ 1920] → '␣process'
   Token 15: [13057] → '␣vast'
   Token 16: [15055] → '␣amounts'
   Token 17: [  315] → '␣of'
   Token 18: [  828] → '␣data'
   Token 19: [  323] → '␣and'
   Token 20: [ 1304] → '␣make'
   ... và 20 tokens nữa


### 3.4 Thử nghiệm với tiếng Việt

In [9]:
# Ví dụ tiếng Việt đơn giản
vietnamese_text = "Xin chào, bạn khỏe không?"

print("=" * 70)
print("THỬ NGHIỆM TIẾNG VIỆT")
print("=" * 70)
result_vi = display_tokens(vietnamese_text, encoding)

THỬ NGHIỆM TIẾNG VIỆT
📝 Văn bản gốc: Xin chào, bạn khỏe không?

📊 Thống kê:
   - Số ký tự: 25
   - Số từ: 5
   - Số tokens: 12
   - Trung bình: 2.08 ký tự/token

🔤 Chi tiết tokens (hiển thị 12 tokens đầu):
   Token 1: [   55] → 'X'
   Token 2: [  258] → 'in'
   Token 3: [  523] → '␣ch'
   Token 4: [ 6496] → 'à'
   Token 5: [   78] → 'o'
   Token 6: [   11] → ','
   Token 7: [90537] → '␣bạn'
   Token 8: [24040] → '␣kh'
   Token 9: [86242] → 'ỏ'
   Token 10: [   68] → 'e'
   Token 11: [54137] → '␣không'
   Token 12: [   30] → '?'


In [10]:
# Ví dụ tiếng Việt dài hơn
vietnamese_paragraph = """
Trí tuệ nhân tạo đang thay đổi thế giới. Các mô hình học máy có thể xử lý 
lượng dữ liệu khổng lồ và đưa ra dự đoán với độ chính xác đáng kinh ngạc. 
Các mô hình ngôn ngữ lớn như GPT có thể hiểu và tạo ra văn bản giống con người.
"""

print("=" * 70)
print("THỬ NGHIỆM TIẾNG VIỆT - ĐOẠN VĂN DÀI")
print("=" * 70)
result_vi_long = display_tokens(vietnamese_paragraph.strip(), encoding)

THỬ NGHIỆM TIẾNG VIỆT - ĐOẠN VĂN DÀI
📝 Văn bản gốc: Trí tuệ nhân tạo đang thay đổi thế giới. Các mô hình học máy có thể xử lý 
lượng dữ liệu khổng lồ và...

📊 Thống kê:
   - Số ký tự: 229
   - Số từ: 54
   - Số tokens: 113
   - Trung bình: 2.03 ký tự/token

🔤 Chi tiết tokens (hiển thị 20 tokens đầu):
   Token 1: [ 1305] → 'Tr'
   Token 2: [ 2483] → 'í'
   Token 3: [ 9964] → '␣tu'
   Token 4: [26298] → 'ệ'
   Token 5: [20921] → '␣nh'
   Token 6: [40492] → 'ân'
   Token 7: [  259] → '␣t'
   Token 8: [89416] → 'ạo'
   Token 9: [15199] → '␣đ'
   Token 10: [  526] → 'ang'
   Token 11: [  270] → '␣th'
   Token 12: [  352] → 'ay'
   Token 13: [15199] → '␣đ'
   Token 14: [98616] → 'ổi'
   Token 15: [  270] → '␣th'
   Token 16: [27160] → 'ế'
   Token 17: [13845] → '␣gi'
   Token 18: [53047] → 'ới'
   Token 19: [   13] → '.'
   Token 20: [  356] → '␣C'
   ... và 93 tokens nữa


### 3.5 So sánh tiếng Anh vs tiếng Việt

In [ ]:
# So sánh các đoạn văn tương tự
comparison_data = {
    'Ngôn ngữ': ['Tiếng Anh', 'Tiếng Việt'],
    'Số ký tự': [result_en_long['num_characters'], result_vi_long['num_characters']],
    'Số từ': [result_en_long['num_words'], result_vi_long['num_words']],
    'Số tokens': [result_en_long['num_tokens'], result_vi_long['num_tokens']],
    'Ký tự/token': [result_en_long['chars_per_token'], result_vi_long['chars_per_token']]
}

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "=" * 70)
print("SO SÁNH TIẾNG ANH VS TIẾNG VIỆT")
print("=" * 70)
print(df_comparison.to_string(index=False))

# Tính tỷ lệ
token_ratio = result_vi_long['num_tokens'] / result_en_long['num_tokens']
print(f"\n💡 Nhận xét: Tiếng Việt tốn nhiều hơn {token_ratio:.2f}x tokens so với tiếng Anh!")

## 💰 Phần 4: Tính toán chi phí API

### 4.1 Bảng giá các model phổ biến (tính đến 2024)

| Model | Input ($/1M tokens) | Output ($/1M tokens) |
|-------|---------------------|----------------------|
| GPT-4 Turbo | $10 | $30 |
| GPT-4 | $30 | $60 |
| GPT-3.5 Turbo | $0.50 | $1.50 |
| Claude 3 Opus | $15 | $75 |
| Claude 3 Sonnet | $3 | $15 |

In [11]:
# Định nghĩa bảng giá
PRICING = {
    'gpt-4-turbo': {'input': 10.0, 'output': 30.0},
    'gpt-4': {'input': 30.0, 'output': 60.0},
    'gpt-3.5-turbo': {'input': 0.5, 'output': 1.5},
    'claude-3-opus': {'input': 15.0, 'output': 75.0},
    'claude-3-sonnet': {'input': 3.0, 'output': 15.0}
}

def calculate_cost(input_tokens: int, output_tokens: int, model: str = 'gpt-4-turbo') -> dict:
    """
    Tính toán chi phí sử dụng API
    
    Args:
        input_tokens: Số tokens đầu vào (prompt)
        output_tokens: Số tokens đầu ra (response)
        model: Tên model sử dụng
    
    Returns:
        Dictionary chứa thông tin chi phí
    """
    if model not in PRICING:
        raise ValueError(f"Model {model} không có trong bảng giá")
    
    pricing = PRICING[model]
    
    # Tính chi phí (giá được tính trên 1 triệu tokens)
    input_cost = (input_tokens / 1_000_000) * pricing['input']
    output_cost = (output_tokens / 1_000_000) * pricing['output']
    total_cost = input_cost + output_cost
    
    return {
        'model': model,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': input_tokens + output_tokens,
        'input_cost': input_cost,
        'output_cost': output_cost,
        'total_cost': total_cost
    }

def display_cost(cost_info: dict):
    """
    Hiển thị thông tin chi phí
    """
    print(f"💰 Chi phí sử dụng {cost_info['model'].upper()}:")
    print(f"   - Input tokens: {cost_info['input_tokens']:,} tokens = ${cost_info['input_cost']:.6f}")
    print(f"   - Output tokens: {cost_info['output_tokens']:,} tokens = ${cost_info['output_cost']:.6f}")
    print(f"   - Tổng: {cost_info['total_tokens']:,} tokens = ${cost_info['total_cost']:.6f}")

### 4.2 Ví dụ tính chi phí cho một cuộc hội thoại

In [12]:
# Giả sử một cuộc hội thoại
user_prompt = "Hãy giải thích về trí tuệ nhân tạo một cách chi tiết và dễ hiểu."
ai_response = """
Trí tuệ nhân tạo (AI) là lĩnh vực khoa học máy tính tập trung vào việc tạo ra các hệ thống 
có khả năng thực hiện các nhiệm vụ đòi hỏi trí thông minh của con người. AI bao gồm nhiều 
kỹ thuật như học máy, học sâu, xử lý ngôn ngữ tự nhiên, và thị giác máy tính. Các ứng dụng 
của AI rất đa dạng, từ nhận diện giọng nói, dịch thuật, xe tự lái, đến chẩn đoán y khoa.
"""

# Đếm tokens
input_tokens = len(encoding.encode(user_prompt))
output_tokens = len(encoding.encode(ai_response))

print("=" * 70)
print("VÍ DỤ: TÍNH CHI PHÍ CHO MỘT CUỘC HỘI THOẠI")
print("=" * 70)
print(f"\n👤 User prompt: {user_prompt}")
print(f"🤖 AI response: {ai_response.strip()[:100]}...\n")

# Tính chi phí cho các model khác nhau
for model_name in ['gpt-3.5-turbo', 'gpt-4-turbo', 'gpt-4']:
    cost = calculate_cost(input_tokens, output_tokens, model_name)
    display_cost(cost)
    print()

VÍ DỤ: TÍNH CHI PHÍ CHO MỘT CUỘC HỘI THOẠI

👤 User prompt: Hãy giải thích về trí tuệ nhân tạo một cách chi tiết và dễ hiểu.
🤖 AI response: Trí tuệ nhân tạo (AI) là lĩnh vực khoa học máy tính tập trung vào việc tạo ra các hệ thống 
có khả n...

💰 Chi phí sử dụng GPT-3.5-TURBO:
   - Input tokens: 33 tokens = $0.000017
   - Output tokens: 186 tokens = $0.000279
   - Tổng: 219 tokens = $0.000296

💰 Chi phí sử dụng GPT-4-TURBO:
   - Input tokens: 33 tokens = $0.000330
   - Output tokens: 186 tokens = $0.005580
   - Tổng: 219 tokens = $0.005910

💰 Chi phí sử dụng GPT-4:
   - Input tokens: 33 tokens = $0.000990
   - Output tokens: 186 tokens = $0.011160
   - Tổng: 219 tokens = $0.012150



### 4.3 So sánh chi phí tiếng Anh vs tiếng Việt

In [14]:
# Sử dụng các đoạn văn đã phân tích ở trên
model = 'gpt-4-turbo'

# Giả sử cả 2 đều có response tương tự nhau (100 tokens)
output_tokens_assumed = 100

cost_en = calculate_cost(result_en_long['num_tokens'], output_tokens_assumed, model)
cost_vi = calculate_cost(result_vi_long['num_tokens'], output_tokens_assumed, model)

print("=" * 70)
print(f"SO SÁNH CHI PHÍ TIẾNG ANH VS TIẾNG VIỆT ({model.upper()})")
print("=" * 70)
print("\nTIẾNG ANH:")
display_cost(cost_en)

print("\nTIẾNG VIỆT:")
display_cost(cost_vi)

cost_increase = ((cost_vi['total_cost'] - cost_en['total_cost']) / cost_en['total_cost']) * 100
print(f"\n💡 Chi phí tiếng Việt cao hơn {cost_increase:.1f}% so với tiếng Anh!")

SO SÁNH CHI PHÍ TIẾNG ANH VS TIẾNG VIỆT (GPT-4-TURBO)

TIẾNG ANH:
💰 Chi phí sử dụng GPT-4-TURBO:
   - Input tokens: 40 tokens = $0.000400
   - Output tokens: 100 tokens = $0.003000
   - Tổng: 140 tokens = $0.003400

TIẾNG VIỆT:
💰 Chi phí sử dụng GPT-4-TURBO:
   - Input tokens: 113 tokens = $0.001130
   - Output tokens: 100 tokens = $0.003000
   - Tổng: 213 tokens = $0.004130

💡 Chi phí tiếng Việt cao hơn 21.5% so với tiếng Anh!


### 4.4 Tính chi phí cho quy mô lớn

In [15]:
# Giả sử bạn cần xử lý 1000 requests mỗi ngày
daily_requests = 1000
avg_input_tokens = 500  # Trung bình mỗi request
avg_output_tokens = 300
days_per_month = 30

# Tính chi phí tháng cho các model
print("=" * 70)
print("CHI PHÍ DỰ KIẾN HÀNG THÁNG")
print("=" * 70)
print(f"Giả định: {daily_requests:,} requests/ngày, {avg_input_tokens} input tokens, {avg_output_tokens} output tokens\n")

results = []
for model_name in PRICING.keys():
    daily_cost = calculate_cost(avg_input_tokens, avg_output_tokens, model_name)['total_cost'] * daily_requests
    monthly_cost = daily_cost * days_per_month
    results.append({
        'Model': model_name,
        'Chi phí/ngày': f"${daily_cost:.2f}",
        'Chi phí/tháng': f"${monthly_cost:.2f}"
    })

df_costs = pd.DataFrame(results)
print(df_costs.to_string(index=False))

CHI PHÍ DỰ KIẾN HÀNG THÁNG
Giả định: 1,000 requests/ngày, 500 input tokens, 300 output tokens

          Model Chi phí/ngày Chi phí/tháng
    gpt-4-turbo       $14.00       $420.00
          gpt-4       $33.00       $990.00
  gpt-3.5-turbo        $0.70        $21.00
  claude-3-opus       $30.00       $900.00
claude-3-sonnet        $6.00       $180.00


## 🎯 Phần 5: BÀI TẬP THỰC HÀNH

### Bài tập 1: Phân tích văn bản của bạn
Hãy thử tokenize một đoạn văn bản tự chọn (tiếng Việt hoặc tiếng Anh) và phân tích kết quả.

In [ ]:
# TODO: Học viên điền đoạn văn của mình vào đây
my_text = """
Viết đoạn văn của bạn vào đây...
"""

# Phân tích
my_result = display_tokens(my_text.strip(), encoding)

### Bài tập 2: So sánh 2 cách viết prompt
Viết 2 prompt khác nhau để đạt được cùng một mục đích. So sánh số tokens và chi phí.

In [ ]:
# TODO: Viết 2 prompt khác nhau
prompt_1 = """Viết prompt dài, chi tiết..."""

prompt_2 = """Viết prompt ngắn gọn hơn nhưng vẫn đạt mục đích..."""

# Phân tích
print("PROMPT 1:")
result_1 = display_tokens(prompt_1, encoding)

print("\n" + "="*70 + "\n")
print("PROMPT 2:")
result_2 = display_tokens(prompt_2, encoding)

# So sánh
print("\n" + "="*70)
print("SO SÁNH:")
print(f"Prompt 1: {result_1['num_tokens']} tokens")
print(f"Prompt 2: {result_2['num_tokens']} tokens")
print(f"Tiết kiệm: {result_1['num_tokens'] - result_2['num_tokens']} tokens ({((result_1['num_tokens'] - result_2['num_tokens'])/result_1['num_tokens']*100):.1f}%)")

### Bài tập 3: Tính toán dự án thực tế
Giả sử bạn đang xây dựng một chatbot. Hãy tính toán chi phí dự kiến.

In [ ]:
# TODO: Điền thông tin dự án của bạn
project_info = {
    'daily_users': 100,  # Số người dùng mỗi ngày
    'messages_per_user': 10,  # Số tin nhắn trung bình mỗi người
    'avg_input_tokens': 50,  # Token trung bình mỗi input
    'avg_output_tokens': 100,  # Token trung bình mỗi output
    'model': 'gpt-3.5-turbo'  # Model sử dụng
}

# Tính toán
total_daily_messages = project_info['daily_users'] * project_info['messages_per_user']
cost_per_message = calculate_cost(
    project_info['avg_input_tokens'],
    project_info['avg_output_tokens'],
    project_info['model']
)['total_cost']

daily_cost = cost_per_message * total_daily_messages
monthly_cost = daily_cost * 30
yearly_cost = daily_cost * 365

print("=" * 70)
print("DỰ TOÁN CHI PHÍ DỰ ÁN")
print("=" * 70)
print(f"Model: {project_info['model']}")
print(f"Số người dùng/ngày: {project_info['daily_users']:,}")
print(f"Tin nhắn/người: {project_info['messages_per_user']}")
print(f"Tổng tin nhắn/ngày: {total_daily_messages:,}")
print(f"\nChi phí ước tính:")
print(f"   - Mỗi tin nhắn: ${cost_per_message:.6f}")
print(f"   - Mỗi ngày: ${daily_cost:.2f}")
print(f"   - Mỗi tháng: ${monthly_cost:.2f}")
print(f"   - Mỗi năm: ${yearly_cost:.2f}")

### Bài tập 4: Tối ưu hóa prompt
Cho prompt dưới đây. Hãy tối ưu để giảm số tokens nhưng vẫn giữ nguyên ý nghĩa.

In [ ]:
# Prompt gốc (chưa tối ưu)
original_prompt = """
Bạn là một trợ lý AI thông minh và hữu ích. Nhiệm vụ của bạn là giúp người dùng 
trả lời các câu hỏi một cách chi tiết và chính xác. Hãy đảm bảo rằng câu trả lời 
của bạn rõ ràng, dễ hiểu và cung cấp đủ thông tin cần thiết. Nếu bạn không chắc 
chắn về câu trả lời, hãy thành thật nói rằng bạn không biết thay vì đưa ra thông 
tin sai lệch.

Câu hỏi của người dùng: Trí tuệ nhân tạo là gì?
"""

# TODO: Viết lại prompt tối ưu hơn
optimized_prompt = """
Viết lại prompt ngắn gọn hơn ở đây...
"""

# So sánh
print("PROMPT GỐC:")
result_original = analyze_text(original_prompt.strip(), encoding)
print(f"Tokens: {result_original['num_tokens']}")

print("\nPROMPT TỐI ƯU:")
result_optimized = analyze_text(optimized_prompt.strip(), encoding)
print(f"Tokens: {result_optimized['num_tokens']}")

saved_tokens = result_original['num_tokens'] - result_optimized['num_tokens']
saved_percentage = (saved_tokens / result_original['num_tokens']) * 100

print(f"\n✅ Tiết kiệm: {saved_tokens} tokens ({saved_percentage:.1f}%)")

### Bài tập 5: Phân tích các loại ký tự đặc biệt
Thử nghiệm tokenization với các loại ký tự khác nhau: số, emoji, ký tự đặc biệt, code...

In [ ]:
# TODO: Thử các loại văn bản khác nhau
test_cases = {
    'Số': '1234567890',
    'Emoji': '😀😃😄😁🤣😂',
    'Code Python': 'def hello(): print("Hello World")',
    'JSON': '{"name": "John", "age": 30}',
    'URL': 'https://www.example.com/path/to/page?param=value',
    'Email': 'user@example.com',
}

print("=" * 70)
print("PHÂN TÍCH CÁC LOẠI KÝ TỰ ĐẶC BIỆT")
print("=" * 70)

for category, text in test_cases.items():
    result = analyze_text(text, encoding)
    print(f"\n{category}:")
    print(f"   Text: {text}")
    print(f"   Tokens: {result['num_tokens']} | Chars/Token: {result['chars_per_token']}")

## 📝 Phần 6: Tổng kết và Lưu ý

### Những điều cần nhớ:

1. **Tiếng Việt tốn nhiều tokens hơn tiếng Anh** (gấp 2-3 lần)
   - Do tokenizer được train chủ yếu trên văn bản tiếng Anh
   - Các ký tự có dấu (á, ă, ơ...) thường được chia thành nhiều tokens

2. **Chi phí = Input tokens + Output tokens**
   - Output tokens thường đắt hơn input tokens (gấp 2-3 lần)
   - Cần tính toán cả 2 phần khi dự toán

3. **Cách tối ưu chi phí:**
   - Viết prompt ngắn gọn, súc tích
   - Sử dụng system message hiệu quả
   - Giới hạn độ dài response (max_tokens)
   - Chọn model phù hợp với nhu cầu (không nhất thiết phải dùng GPT-4)
   - Cache các response thường dùng
   - Batch processing khi có thể

4. **Context window:**
   - Mỗi model có giới hạn tokens khác nhau
   - GPT-3.5-turbo: 4K-16K tokens
   - GPT-4-turbo: 128K tokens
   - Cần tính cả input + output

### Bài tập về nhà:
1. Phân tích một tài liệu thực tế của bạn
2. So sánh chi phí các model khác nhau cho use case cụ thể
3. Tối ưu hóa prompt để giảm 30% tokens
4. Tính toán ROI cho một dự án AI

## 🔗 Phần 7: Tài nguyên tham khảo

- [OpenAI Tokenizer](https://platform.openai.com/tokenizer)
- [OpenAI Pricing](https://openai.com/pricing)
- [Tiktoken Documentation](https://github.com/openai/tiktoken)
- [Best Practices for Prompt Engineering](https://platform.openai.com/docs/guides/prompt-engineering)

---
**Chúc các bạn học tốt! 🎓**